# 02 — Model Evaluation

Systematic post-training analysis of the YOLO-OBB boat detection model.

All metric computation is delegated to `src.evaluation.metrics`.
This notebook contains only configuration constants and visualisation code.

| § | Description |
|---|-------------|
| 1 | Setup & constants |
| 2 | Model loading & formal val() metrics |
| 3 | Per-class performance table |
| 4 | Normalised confusion matrix |
| 5 | Dataset statistics |
| 6 | OBB geometric statistics |
| 7 | PR curve |
| 8 | TP / FP / FN detection crops |

In [ ]:
# ── Standard library ──────────────────────────────────────────────────────
import sys, logging
from pathlib import Path

sys.path.insert(0, str(Path.cwd().parent))

# ── Third-party ───────────────────────────────────────────────────────────
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns
import cv2

# ── Project ───────────────────────────────────────────────────────────────
from src.utils.config import load_config
from src.evaluation.metrics import (
    compute_per_class_metrics,
    compute_confusion_matrix,
    build_size_dataframe,
    build_distribution_dataframe,
    parse_label_file,
    match_detections,
    compute_crop_metrics,
    build_crop_summary,
)

logging.getLogger("ultralytics").setLevel(logging.ERROR)

plt.rcParams.update({
    "figure.dpi": 150,
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "axes.spines.top": False,
    "axes.spines.right": False,
    "axes.grid": True,
    "grid.alpha": 0.35,
    "legend.framealpha": 0.7,
    "legend.fontsize": 9,
})
print("Imports OK.")

## 1. Constants

Change these to adapt to a different run or dataset.

In [ ]:
MODEL_PATH    = Path("../runs/obb/boat_obb_v1/weights/best.pt")
DATA_PATH     = Path("../data/dataset.yaml")
PROCESSED_DIR = Path("../data/processed")
LABELS_ROOT   = PROCESSED_DIR / "labels"
IMAGES_DIR    = PROCESSED_DIR / "images"
SPLITS        = ["train", "val", "test"]

FINAL_CONF    = 0.10
FINAL_IOU     = 0.30
IMG_SIZE      = 640      # must match training imgsz
TILE_SIZE     = 640      # tile size used in preprocessing

print(f"Model  : {MODEL_PATH}")
print(f"Data   : {DATA_PATH}")
print(f"Conf   : {FINAL_CONF}  |  IoU : {FINAL_IOU}")

## 2. Model Loading

In [ ]:
from ultralytics import YOLO

model       = YOLO(MODEL_PATH)
CLASS_NAMES = model.names
print("Classes:", CLASS_NAMES)

## 3. Formal Evaluation — model.val()

In [ ]:
metrics = model.val(
    data=str(DATA_PATH),
    split="test",
    imgsz=IMG_SIZE,
    conf=FINAL_CONF,
    iou=FINAL_IOU,
    save_json=False,
    plots=False,
    verbose=False,
)

print(f"mAP50    : {metrics.box.map50:.4f}")
print(f"mAP50-95 : {metrics.box.map:.4f}")
print(f"Precision: {metrics.box.mp:.4f}")
print(f"Recall   : {metrics.box.mr:.4f}")

## 4. Per-Class Performance Table

In [ ]:
df_perf = compute_per_class_metrics(metrics, CLASS_NAMES, LABELS_ROOT, test_split="test")
display(df_perf)

## 5. Normalised Confusion Matrix

In [ ]:
metrics_cm = model.val(
    data=str(DATA_PATH), split="test",
    imgsz=IMG_SIZE, conf=FINAL_CONF, iou=FINAL_IOU,
    plots=True, save_json=False, verbose=False,
)

cm_norm, labels = compute_confusion_matrix(metrics_cm, CLASS_NAMES)

fig, ax = plt.subplots(figsize=(9, 8))
im = ax.imshow(cm_norm, vmin=0, vmax=1, cmap="Blues")
plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

ticks = np.arange(len(labels))
ax.set_xticks(ticks); ax.set_xticklabels(labels, rotation=40, ha="right")
ax.set_yticks(ticks); ax.set_yticklabels(labels)

thresh = cm_norm.max() / 2.0
for i in range(cm_norm.shape[0]):
    for j in range(cm_norm.shape[1]):
        v = cm_norm[i, j]
        if not np.isnan(v):
            ax.text(j, i, f"{v:.2f}", ha="center", va="center",
                    fontsize=8, color="white" if v > thresh else "black")

ax.set_ylabel("True label"); ax.set_xlabel("Predicted label")
ax.set_title(f"Normalised Confusion Matrix (conf={FINAL_CONF}, IoU={FINAL_IOU})",
             fontweight="bold")
plt.tight_layout()
plt.show()

## 6. Dataset Statistics

In [ ]:
df_dist = build_distribution_dataframe(LABELS_ROOT, SPLITS, CLASS_NAMES)
print("Object counts per class and split:")
display(df_dist)

classes = list(CLASS_NAMES.values())
x, width = np.arange(len(classes)), 0.25

fig, ax = plt.subplots(figsize=(12, 5))
for i, split in enumerate(SPLITS):
    col = split.capitalize()
    vals = df_dist[col].values if col in df_dist.columns else [0]*len(classes)
    ax.bar(x + i * width, vals, width, label=col)

ax.set_xticks(x + width)
ax.set_xticklabels(classes, rotation=30, ha="right")
ax.set_ylabel("Object count")
ax.set_title("Class Distribution per Split", fontweight="bold")
ax.legend(); plt.tight_layout(); plt.show()

## 7. OBB Geometric Statistics

Long side, short side, area, and aspect ratio per class across all splits.

In [ ]:
df_sizes = build_size_dataframe(LABELS_ROOT, SPLITS, CLASS_NAMES, tile_size=TILE_SIZE)
display(df_sizes)

fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for ax, col, title in zip(
    axes,
    ["Long mean (px)", "Short mean (px)", "Aspect ratio"],
    ["Mean Long Side (px)", "Mean Short Side (px)", "Mean Aspect Ratio"],
):
    ax.barh(df_sizes["Class"], df_sizes[col], color="steelblue")
    ax.set_xlabel(col); ax.set_title(title, fontweight="bold")

plt.tight_layout(); plt.show()

## 8. PR Curve

Sweep confidence threshold and record precision/recall to plot the PR curve.

In [ ]:
thresholds  = np.linspace(0.05, 0.95, 40)
pr_records  = []

images_test = sorted((IMAGES_DIR / "test").glob("*.tif"))
print(f"Running PR sweep over {len(thresholds)} thresholds on {len(images_test)} tiles …")

for conf in thresholds:
    tp = fp = fn = 0
    for img_path in images_test:
        import rasterio as _rio
        with _rio.open(img_path) as ds:
            w, h = ds.width, ds.height

        lbl   = LABELS_ROOT / "test" / (img_path.stem + ".txt")
        gt    = parse_label_file(lbl, w, h)
        res   = model.predict(str(img_path), imgsz=IMG_SIZE,
                              conf=float(conf), iou=FINAL_IOU, verbose=False)[0]

        preds = []
        if res.obb is not None and len(res.obb):
            for pts, cid, cf in zip(
                res.obb.xyxyxyxy.cpu().numpy(),
                res.obb.cls.cpu().numpy().astype(int),
                res.obb.conf.cpu().numpy(),
            ):
                preds.append((int(cid), pts.astype(np.float32), float(cf)))

        m_pred, m_gt = match_detections(gt, preds, iou_threshold=FINAL_IOU)
        tp += sum(m_pred)
        fp += sum(not x for x in m_pred)
        fn += sum(not x for x in m_gt)

    p, r, f1 = compute_crop_metrics(tp, fp, fn)
    pr_records.append({"conf": conf, "precision": p, "recall": r, "f1": f1})

df_pr = pd.DataFrame(pr_records)

fig, axes = plt.subplots(1, 2, figsize=(13, 5))

ax = axes[0]
ax.plot(df_pr["recall"], df_pr["precision"], "b-o", ms=4, label="P-R curve")
ax.set_xlabel("Recall"); ax.set_ylabel("Precision")
ax.set_xlim(0, 1); ax.set_ylim(0, 1)
ax.set_title("Precision-Recall Curve", fontweight="bold")
best = df_pr.loc[df_pr["f1"].idxmax()]
ax.annotate(f'F1={best.f1:.3f}\nconf={best.conf:.2f}',
            xy=(best.recall, best.precision), fontsize=9,
            xytext=(best.recall - 0.2, best.precision - 0.12),
            arrowprops=dict(arrowstyle="->")); ax.legend()

ax = axes[1]
ax.plot(df_pr["conf"], df_pr["precision"], label="Precision")
ax.plot(df_pr["conf"], df_pr["recall"],    label="Recall")
ax.plot(df_pr["conf"], df_pr["f1"],        label="F1", linestyle="--")
ax.set_xlabel("Confidence threshold"); ax.set_ylabel("Score")
ax.set_title("Metrics vs Confidence Threshold", fontweight="bold")
ax.legend()

plt.tight_layout(); plt.show()

## 9. Detection Crop Export — TP / FP / FN

Crops are saved to `crops/<TruePositive|FalsePositive|FalseNegative>/<class>/`.
A summary CSV is written to `crops/summary.csv`.

In [ ]:
IOU_MATCH  = FINAL_IOU
CROP_PAD   = 80
CROPS_ROOT = Path("../crops")
SPLIT      = "test"

OBB_COLOR     = (0, 0, 255)
OBB_THICKNESS = 2

for sub in ("TruePositive", "FalsePositive", "FalseNegative"):
    (CROPS_ROOT / sub).mkdir(parents=True, exist_ok=True)


def crop_obb(img_bgr, pts_px, pad=CROP_PAD):
    h, w = img_bgr.shape[:2]
    x1 = max(0,     int(pts_px[:, 0].min()) - pad)
    y1 = max(0,     int(pts_px[:, 1].min()) - pad)
    x2 = min(w - 1, int(pts_px[:, 0].max()) + pad)
    y2 = min(h - 1, int(pts_px[:, 1].max()) + pad)
    return img_bgr[y1:y2, x1:x2].copy(), (x1, y1)

def draw_obb(img, pts_px, origin=(0, 0)):
    shifted = pts_px.copy()
    shifted[:, 0] -= origin[0]; shifted[:, 1] -= origin[1]
    cv2.polylines(img, [shifted.astype(np.int32).reshape(-1, 1, 2)],
                  True, OBB_COLOR, OBB_THICKNESS)
    return img


summary_rows = []
tp_count = fp_count = fn_count = 0

for img_path in sorted((IMAGES_DIR / SPLIT).glob("*.tif")):
    import rasterio as _rio
    with _rio.open(img_path) as ds:
        rgb_data = ds.read()
    img_bgr = cv2.cvtColor(rgb_data[:3].transpose(1, 2, 0), cv2.COLOR_RGB2BGR)
    h, w = img_bgr.shape[:2]
    stem  = img_path.stem

    lbl = LABELS_ROOT / SPLIT / (stem + ".txt")
    gt  = parse_label_file(lbl, w, h)

    res   = model.predict(str(img_path), imgsz=IMG_SIZE,
                          conf=FINAL_CONF, iou=FINAL_IOU, verbose=False)[0]
    preds = []
    if res.obb is not None and len(res.obb):
        for pts, cid, cf in zip(
            res.obb.xyxyxyxy.cpu().numpy(),
            res.obb.cls.cpu().numpy().astype(int),
            res.obb.conf.cpu().numpy(),
        ):
            preds.append((int(cid), pts.astype(np.float32), float(cf)))

    m_pred, m_gt = match_detections(gt, preds, IOU_MATCH)

    # TPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if m_pred[pi]:
            crop, origin = crop_obb(img_bgr, pr_pts)
            if crop.size:
                crop = draw_obb(crop, pr_pts, origin)
                out  = CROPS_ROOT / "TruePositive" / CLASS_NAMES[pr_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__tp__{tp_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "TruePositive",
                                     "class": CLASS_NAMES[pr_cid], "index": tp_count})
                tp_count += 1

    # FPs
    for pi, (pr_cid, pr_pts, pr_cf) in enumerate(preds):
        if not m_pred[pi]:
            out = CROPS_ROOT / "FalsePositive" / CLASS_NAMES[pr_cid]
            out.mkdir(parents=True, exist_ok=True)
            canvas = img_bgr.copy()
            draw_obb(canvas, pr_pts)
            cv2.imwrite(str(out / f"{stem}__fp__{fp_count:04d}.png"), canvas)
            summary_rows.append({"image": stem, "category": "FalsePositive",
                                 "class": CLASS_NAMES[pr_cid], "index": fp_count})
            fp_count += 1

    # FNs
    for gi, (gt_cid, gt_pts) in enumerate(gt):
        if not m_gt[gi]:
            crop, _ = crop_obb(img_bgr, gt_pts, pad=CROP_PAD)
            if crop.size:
                out = CROPS_ROOT / "FalseNegative" / CLASS_NAMES[gt_cid]
                out.mkdir(parents=True, exist_ok=True)
                cv2.imwrite(str(out / f"{stem}__fn__{fn_count:04d}.png"), crop)
                summary_rows.append({"image": stem, "category": "FalseNegative",
                                     "class": CLASS_NAMES[gt_cid], "index": fn_count})
                fn_count += 1

df_crops = build_crop_summary(summary_rows)
if not df_crops.empty:
    df_crops.to_csv(CROPS_ROOT / "summary.csv", index=False)

p, r, f1 = compute_crop_metrics(tp_count, fp_count, fn_count)
print(f"TP: {tp_count}  FP: {fp_count}  FN: {fn_count}")
print(f"Precision: {p:.4f}  Recall: {r:.4f}  F1: {f1:.4f}")
print(f"Crops saved to: {CROPS_ROOT.resolve()}")
display(df_crops)